# Consignes

Intégrer le fichier USvideos.csv qui représente un ensemble de 8000 vidéos Youtube. 

Merger le fichier US_category_id.json pour récupérer le nom des catégories. Il conviendra de bien spécifier l'ID du document.


# Questions 
- 1) Récupérer toutes les vidéos de la chaîne Apple.
- 2) Compter le nombre de catégories différentes
- 3) Si vous ne l'avez pas déjà fait, découper les tags en listes et mettre à jour les tags de chacun des documents avec une requête update.
- 4) Récupérer les vidéos les plus vues.
- 5) Compter le nombre moyen de vues en fonction de la catégorie.
- 6) Récupérer les chaines Youtube avec la plus grande moyenne de likes.

In [100]:
import pymongo

In [101]:
import pandas as pd

In [102]:
client = pymongo.MongoClient()
database = client['exercices']
collection = database['youtube']

In [103]:
df_youtube = pd.read_csv("./data/USvideos.csv")
df_youtube.head(2)
df_youtube.head()


,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,13.09
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,13.09
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,13.09
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,13.09
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,13.09


## Question 0

### Netoyer les données

In [104]:
import json

with open('/Users/darius/DataEngeniring/DataEngineerTools/3Mongo/data/US_category_id.json', 'r') as f:
    data = json.load(f)

if 'items' in data:
    data = data['items']
    
df_categories = pd.json_normalize(data)
print(df_categories.columns)




Index(['kind', 'etag', 'id', 'snippet.channelId', 'snippet.title',
       'snippet.assignable'],
      dtype='object')


In [105]:
df_categories = df_categories[['id', 'snippet.title']]
df_categories.columns = ['category_id', 'category_name']


In [106]:
df_categories['category_id'] = df_categories['category_id'].astype(str)

In [107]:

df_youtube['category_id'] = df_youtube['category_id'].astype(str)

# La fusion
df_final = pd.merge(
    df_youtube,
    df_categories,
    left_on='category_id',  # Colonne dans vos vidéos
    right_on='category_id', # Colonne dans le fichier json qu'on vient de charger
    how='left'
)



### Importer les données

In [108]:
df_youtube.info()
df_youtube.head()
df_final.info()
df_final.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7992 entries, 0 to 7991
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7992 non-null   object 
 1   title           7992 non-null   object 
 2   channel_title   7992 non-null   object 
 3   category_id     7992 non-null   object 
 4   tags            7992 non-null   object 
 5   views           7992 non-null   int64  
 6   likes           7992 non-null   int64  
 7   dislikes        7992 non-null   int64  
 8   comment_total   7992 non-null   int64  
 9   thumbnail_link  7992 non-null   object 
 10  date            7992 non-null   float64
dtypes: float64(1), int64(4), object(6)
memory usage: 686.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7992 entries, 0 to 7991
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7992 non-null   object 
 1

,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date,category_name
0,XpVt6Z1Gjjo,1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...,Logan Paul Vlogs,24,logan paul vlog|logan paul|logan|paul|olympics...,4394029,320053,5931,46245,https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg,13.09,Entertainment
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,13.09,Science & Technology
2,cLdxuaxaQwc,My Response,PewDiePie,22,[none],5845909,576597,39774,170708,https://i.ytimg.com/vi/cLdxuaxaQwc/default.jpg,13.09,People & Blogs
3,WYYvHb03Eog,Apple iPhone X first look,The Verge,28,apple iphone x hands on|Apple iPhone X|iPhone ...,2642103,24975,4542,12829,https://i.ytimg.com/vi/WYYvHb03Eog/default.jpg,13.09,Science & Technology
4,sjlHnJvXdQs,iPhone X (parody),jacksfilms,23,jacksfilms|parody|parodies|iphone|iphone x|iph...,1168130,96666,568,6666,https://i.ytimg.com/vi/sjlHnJvXdQs/default.jpg,13.09,Comedy


## Question 1  

In [109]:
df_Apple = df_final[df_final['channel_title'] == 'Apple']
df_Apple.head()


,video_id,title,channel_title,category_id,tags,views,likes,dislikes,comment_total,thumbnail_link,date,category_name
1,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,7860119,185853,26679,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,13.09,Science & Technology
203,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,12200526,258842,44339,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,14.09,Science & Technology
419,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,15731493,321403,57528,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,15.09,Science & Technology
674,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,18082737,359392,64933,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,16.09,Science & Technology
905,K4wEI5zhHB0,iPhone X — Introducing iPhone X — Apple,Apple,28,Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...,19707391,381919,69465,0,https://i.ytimg.com/vi/K4wEI5zhHB0/default.jpg,17.09,Science & Technology


In [110]:
collection.delete_many({})
print("Collection vidée.")

Collection vidée.


In [111]:
data_to_insert = df_final.to_dict(orient='records')
if len(data_to_insert) > 0:
    collection.insert_many(data_to_insert)
    print(f"{len(data_to_insert)} vidéos ajoutées avec succès dans MongoDB !")
else:
    print("Le DataFrame est vide, rien à ajouter.")

7992 vidéos ajoutées avec succès dans MongoDB !


In [112]:
cursor = collection.find().limit(5)
df_visu = pd.DataFrame(list(cursor))
print(df_visu.head())

                        _id     video_id  \
0  692f012547de005184ff82f8  XpVt6Z1Gjjo   
1  692f012547de005184ff82f9  K4wEI5zhHB0   
2  692f012547de005184ff82fa  cLdxuaxaQwc   
3  692f012547de005184ff82fb  WYYvHb03Eog   
4  692f012547de005184ff82fc  sjlHnJvXdQs   

                                               title     channel_title  \
0  1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED Y...  Logan Paul Vlogs   
1            iPhone X — Introducing iPhone X — Apple             Apple   
2                                        My Response         PewDiePie   
3                          Apple iPhone X first look         The Verge   
4                                  iPhone X (parody)        jacksfilms   

  category_id                                               tags    views  \
0          24  logan paul vlog|logan paul|logan|paul|olympics...  4394029   
1          28  Apple|iPhone 10|iPhone Ten|iPhone|Portrait Lig...  7860119   
2          22                                             [

## Question 2

In [113]:
nb_mongo = collection.count_documents({})
nb_pandas = len(df_final)

if nb_mongo == nb_pandas:
    print("✅ Tout est parfait ! Le compte est bon.")
elif nb_mongo > nb_pandas:
    print(f"⚠️ Attention : Il y a plus de documents dans Mongo ({nb_mongo}) que dans le DataFrame ({nb_pandas}). Vous avez probablement lancé l'insertion plusieurs fois (doublons).")
else:
    print("❌ Erreur : Il manque des données.")

✅ Tout est parfait ! Le compte est bon.


In [114]:

liste = [
    {
        "$group": {
            "_id": "$category_name",   
            "total": {"$sum": 1}       
        }
    },
    {
        "$sort": {"total": -1} 
    }
]
resultats = collection.aggregate(liste)
print("--- Répartition par catégorie ---")
for ligne in resultats:
    print(f"{ligne['_id']} : {ligne['total']} vidéos")

--- Répartition par catégorie ---
Entertainment : 1601 vidéos
Music : 1250 vidéos
People & Blogs : 882 vidéos
Howto & Style : 869 vidéos
Comedy : 755 vidéos
News & Politics : 623 vidéos
Science & Technology : 512 vidéos
Sports : 410 vidéos
Film & Animation : 378 vidéos
Education : 334 vidéos
Autos & Vehicles : 116 vidéos
Pets & Animals : 116 vidéos
Gaming : 82 vidéos
Travel & Events : 48 vidéos
Nonprofits & Activism : 14 vidéos
Shows : 2 vidéos


## Question 3

In [115]:
cursor = collection.find({"tags": {"$exists": True}})
count = 0
for i in cursor:
    current_tags = i['tags']
    if isinstance(current_tags, str):
        tag_list = [tag.strip().replace("'", "").replace("[", "").replace("]", "") for tag in current_tags.split(',')]
        collection.update_one(
            {"_id": i["_id"]}, 
            {"$set": {"tags": tag_list}}
        )
        count += 1

print(f"✅ {count} vidéos ont vu leurs tags transformés en liste.")

✅ 7992 vidéos ont vu leurs tags transformés en liste.


## Question 4

In [116]:
documents = collection.find()
for doc in documents:
    updates = {}
    if 'viewCount' in doc and isinstance(doc['viewCount'], str):

        clean_views = ''.join(filter(str.isdigit, doc['viewCount']))
        updates['viewCount'] = int(clean_views) if clean_views else 0
        

    if 'likeCount' in doc and isinstance(doc['likeCount'], str):
        clean_likes = ''.join(filter(str.isdigit, doc['likeCount']))
        updates['likeCount'] = int(clean_likes) if clean_likes else 0
        
    if updates:
        collection.update_one({'_id': doc['_id']}, {'$set': updates})

print("Conversion des chiffres terminée !")

Conversion des chiffres terminée !


In [117]:
# Affiche un document brut pour voir les noms des clés
import pprint
pprint.pprint(collection.find_one())

{'_id': ObjectId('692f012547de005184ff82f8'),
 'category_id': '24',
 'category_name': 'Entertainment',
 'channel_title': 'Logan Paul Vlogs',
 'comment_total': 46245,
 'date': 13.09,
 'dislikes': 5931,
 'likes': 320053,
 'tags': ['logan paul vlog|logan paul|logan|paul|olympics|logan paul '
          'youtube|vlog|daily|comedy|hollywood|parrot|maverick|bird|maverick '
          'clothes|diamond play button|logan paul diamond play button|10M '
          'subscribers|logan paul 1 year vlogging|1 year vlog|dwarf mamba play '
          'button|logan paul history|youtube history|10M|10M plaque|youtube '
          'button|diamond button|logang|logang 4 life'],
 'thumbnail_link': 'https://i.ytimg.com/vi/XpVt6Z1Gjjo/default.jpg',
 'title': '1 YEAR OF VLOGGING -- HOW LOGAN PAUL CHANGED YOUTUBE FOREVER!',
 'video_id': 'XpVt6Z1Gjjo',
 'views': 4394029}


In [118]:
top_videos = collection.find().sort("views", -1).limit(5)

print("--- Top 5 Vidéos ---")
for vid in top_videos:
    print(f"{vid.get('views')} vues : {vid.get('title')}")

--- Top 5 Vidéos ---
41500672 vues : BTS (방탄소년단) 'DNA' Official MV
38013692 vues : BTS (방탄소년단) 'DNA' Official MV
36323498 vues : ZAYN - Dusk Till Dawn ft. Sia
33191594 vues : Eminem Rips Donald Trump In BET Hip Hop Awards Freestyle Cypher
32512343 vues : Eminem Rips Donald Trump In BET Hip Hop Awards Freestyle Cypher


In [119]:
pipeline_top = [
    {                                                                #Convertion en entier
        "$addFields": {                                                                                     
            "views_int": { "$toInt": "$views" } 
        }
    },
    {
        "$group": {
            "_id": "$title",
            "max_vues": { "$max": "$views_int" },
            "chaine": { "$first": "$channel_title" } 
        }
    },

    { "$sort": { "max_vues": -1 } },

    { "$limit": 5 }
]
resultats = list(collection.aggregate(pipeline_top))
print("--- TOP 5 VIDÉOS DE LA MORT QUI TUE---")
for vid in resultats:
    print(f"{vid['max_vues']:,} vues : {vid['_id']}")

--- TOP 5 VIDÉOS DE LA MORT QUI TUE---
41,500,672 vues : BTS (방탄소년단) 'DNA' Official MV
36,323,498 vues : ZAYN - Dusk Till Dawn ft. Sia
33,191,594 vues : Eminem Rips Donald Trump In BET Hip Hop Awards Freestyle Cypher
32,136,948 vues : Shakira - Perro Fiel (Official Video) ft. Nicky Jam
27,909,589 vues : Star Wars: The Last Jedi Trailer (Official)


## Question 5

In [120]:
pipeline_vues = [

    {
        "$addFields": {
            "views_int": { "$toInt": "$views" }  
        }
    },
    {
        "$group": {
            "_id": "$category_name",
            "moyenne_vues": {"$avg": "$views_int"}
        }
    },
    {
        "$sort": {"moyenne_vues": -1}
    }
]

resultats = list(collection.aggregate(pipeline_vues))

print("--- Moyenne de vues par catégorie ---")
for vid in resultats:
    # SECURITE : Si la moyenne est vide, on force à 0
    valeur_moyenne = vid['moyenne_vues'] or 0
    print(f"{int(valeur_moyenne):,} vues : {vid['_id']}")

--- Moyenne de vues par catégorie ---
1,240,073 vues : Comedy
1,176,553 vues : Music
1,154,868 vues : Entertainment
1,110,334 vues : Nonprofits & Activism
1,039,472 vues : Film & Animation
971,532 vues : People & Blogs
924,730 vues : Science & Technology
728,434 vues : Sports
681,081 vues : Gaming
651,404 vues : Pets & Animals
607,693 vues : Autos & Vehicles
547,582 vues : Education
540,955 vues : News & Politics
537,665 vues : Howto & Style
464,041 vues : Travel & Events
8,492 vues : Shows


## Question 6 

In [121]:
pipeline_likes = [
    # 1. On convertit la colonne 'likes' en nombre entier (au cas où c'est du texte)
    {
        "$addFields": {
            "likes_int": { "$toInt": "$likes" } 
        }
    },
    # 2. On regroupe par nom de chaîne (channel_title) et on fait la moyenne
    {
        "$group": {
            "_id": "$channel_title", 
            "moyenne_likes": { "$avg": "$likes_int" }
        }
    },
    # 3. On trie du plus grand au plus petit (-1)
    {
        "$sort": { "moyenne_likes": -1 }
    },
    # 4. On garde le Top 5
    {
        "$limit": 5
    }
]

# Exécution de la requête
resultats_likes = list(collection.aggregate(pipeline_likes))

print("--- TOP 5 CHAÎNES (Moyenne de Likes) ---")
for chaine in resultats_likes:
    # SÉCURITÉ ANTI-CRASH :
    # Si la moyenne est vide (None), on met 0 par défaut
    moy = chaine['moyenne_likes'] or 0
    
    # On affiche proprement
    print(f"{chaine['_id']} : {int(moy):,} likes en moyenne")

--- TOP 5 CHAÎNES (Moyenne de Likes) ---
ZaynVEVO : 1,431,683 likes en moyenne
ibighit : 1,371,766 likes en moyenne
melanie martinez : 911,871 likes en moyenne
BETNetworks : 769,687 likes en moyenne
jypentertainment : 758,826 likes en moyenne


In [122]:
xcnjbvxcnvbxcnv,v


NameError: name 'xcnjbvxcnvbxcnv' is not defined